In [1]:
from experiments.bench import harness, plots
import pandas as pd
# 스윕은 두 번 돌았고 둘 다 커밋돼 있다. 판정은 light path까지 예열한
# 재측정(_primed)으로 하고, 첫 스윕은 두 실행이 같은 결론에 도달했음을 보이는
# 데만 쓴다 — 두 스윕의 절대값을 서로 빼지는 않는다(실행 간 분산이 크다).
result = harness.load_result("exp5_persistence_primed")
first = harness.load_result("exp5_persistence")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cpu_model': 'Apple M4', 'ram': '16.0 GiB', 'cbc': 'Version: 2.10.3', 'neo4j_server': 'Neo4j Kernel 5.26.28 (community)', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
df = pd.DataFrame(result["data"]["rows"])
tbl = df.pivot_table(index=["metric", "n_people"], columns="backend",
                     values="value")
tbl["neo4j/sqlite"] = tbl["neo4j"] / tbl["sqlite"]
display(tbl.round(3))

backend                           neo4j       sqlite  neo4j/sqlite
metric           n_people                                         
append_cowork_ms 100       5.500000e-01        0.231         2.376
                 300       4.890000e-01        0.255         1.920
                 500       7.020000e-01        0.244         2.884
                 1000      5.450000e-01        0.230         2.372
append_review_ms 100       7.350000e-01        0.260         2.826
                 300       4.550000e-01        0.277         1.642
                 500       5.380000e-01        0.241         2.236
                 1000      8.600000e-01        0.344         2.498
disk_bytes       100       1.109951e+09   147456.000      7527.336
                 300       1.112881e+09   356352.000      3122.982
                 500       1.386893e+09   557056.000      2489.683
                 1000      1.397572e+09  1048576.000      1332.828
disk_bytes_clean 100       5.414586e+08   147456.000      3672.001
                 300       5.419255e+08   356352.000      1520.759
                 500       5.424334e+08   557056.000       973.750
                 1000      5.436622e+08  1048576.000       518.477
load_ms          100       5.115000e+01        4.407        11.606
                 300       8.649700e+01        7.366        11.743
                 500       1.294030e+02        9.830        13.164
                 1000      3.386990e+02       17.394        19.472
rehydrate_ms     100       1.516300e+01        3.352         4.524
                 300       3.678300e+01        8.711         4.222
                 500       5.859100e+01       15.145         3.869
                 1000      1.264970e+02       49.294         2.566

In [3]:
plots.exp5_persistence(result)   # figures/exp5_persistence.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp5_persistence.png')

In [4]:
from experiments import decision
from experiments import report as _report
from IPython.display import Markdown, display

d, d0 = result["data"], first["data"]
tally = decision.rule2_tally(d)
tally0 = decision.rule2_tally(d0)
persisted = d["rule2_tally"]["by_disk_reading"][d["rule2_tally"]["adjudicated_on"]]
contradictions = _report.calibration_contradictions(d)
contradictions0 = _report.calibration_contradictions(d0)

lines = ["## 관찰 요약", ""]
lines.append(f"- **의사결정 규칙 2가 소비하는 수치: {tally['metrics']}지표 중 Neo4j "
             f"우위 {tally['neo4j_metric_wins']}개** (disk 판독 "
             f"`{tally['disk_reading']}`) — 사전 고정 임계 "
             f"{decision.PERSISTENCE_THRESHOLD} 미달.")
lines.append("- 지표별: " + ", ".join(
    f"`{m['metric']}` {m['neo4j_wins']}/{m['cells']}셀" for m in tally["per_metric"]))
lines.append(f"- 첫 스윕(`exp5_persistence.json`)도 "
             f"{tally0['neo4j_metric_wins']}/{tally0['metrics']}로 같은 결론이다. "
             "두 실행의 절대값을 서로 빼지는 말 것 — 실행 간 분산이 크다.")
lines.append(f"- 실험 5 러너가 JSON에 남긴 집계와 `decision.evaluate`의 집계가 "
             f"일치한다({persisted['neo4j_metric_wins']}/{persisted['metrics']}) "
             "— 서로 독립인 두 계산이다.")
lines.append(f"- 캘리브레이션이 물리적으로 불가능한 순서를 내는 셀: 판정 스윕 "
             f"{len(contradictions)}건, 첫 스윕 {len(contradictions0)}건 — "
             "`append_*`에 대해서는 차감 기반 주장이 어느 방향으로도 성립하지 않는다.")
lines.append("- 여기서 \"영속성\"은 커밋되어 새 클라이언트에서 보인다는 뜻이다. "
             "크래시 내구성은 양쪽 다 시험하지 않았다.")

display(Markdown("\n".join(lines)))

## 관찰 요약

- **의사결정 규칙 2가 소비하는 수치: 4지표 중 Neo4j 우위 0개** (disk 판독 `disk_bytes_clean`) — 사전 고정 임계 3 미달.
- 지표별: `load_ms` 0/4셀, `disk_bytes` 0/4셀, `append_*_ms` 0/8셀, `rehydrate_ms` 0/4셀
- 첫 스윕(`exp5_persistence.json`)도 0/4로 같은 결론이다. 두 실행의 절대값을 서로 빼지는 말 것 — 실행 간 분산이 크다.
- 실험 5 러너가 JSON에 남긴 집계와 `decision.evaluate`의 집계가 일치한다(0/4) — 서로 독립인 두 계산이다.
- 캘리브레이션이 물리적으로 불가능한 순서를 내는 셀: 판정 스윕 1건, 첫 스윕 2건 — `append_*`에 대해서는 차감 기반 주장이 어느 방향으로도 성립하지 않는다.
- 여기서 "영속성"은 커밋되어 새 클라이언트에서 보인다는 뜻이다. 크래시 내구성은 양쪽 다 시험하지 않았다.